# Multi-Task CNN Training for Out-Of-Distribution Plant Disease Identification

This notebook trains one or more **Multi-Task CNN** configurations (`train/train_multi_task_cnn.py`) sequentially.

### Pipeline Overview:
1. **Configuration & Parameters**: List of experiment configurations (`CONFIG_PATHS`), Google Drive paths, overrides, and options.
2. **Google Drive Integration**: Mounts `/content/drive` to access project folder `Nat_AML_DPD_group_assignment`.
3. **Fast Local Extraction**: Copies `DPD_Small.7z` from Drive to Colab local SSD (`/content/`) and extracts with `7z` to maximize DataLoader throughput.
4. **Environment Setup**: Installs required Python dependencies.
5. **Sequential Multi-Task Training**: Runs 2-stage training for every config in `CONFIG_PATHS` with live Google Drive checkpointing & sync.
6. **Summary & Comparison**: Summarizes all configurations and their results side-by-side.


In [ ]:
# ================= Step 1: Configuration & Experiment Parameters ================
import os
from pathlib import Path

# --- Model & Training Configurations (run sequentially in order) ---
CONFIG_PATHS = [
    "config/multitask/Nat_multitask_convnext_base_v4_1_cutmix.json",
    "config/multitask/Nat_multitask_convnext_base_v4_2_balsoftmax.json",
    "config/multitask/Nat_multitask_convnext_base_v4_3_droppath.json",
]
RESUME = False  # Set to True to resume from last saved checkpoint

# Command-line overrides applied via --set key.path=value
OVERRIDES = [
    "runtime.num_workers=2",                # Recommended for standard Colab vCPUs
    "runtime.checkpoint_every_n_epochs=5",  # Save checkpoint every 5 epochs
]

# --- Google Drive & Sync Paths ---
GDRIVE_PROJECT = Path("/content/drive/MyDrive/Nat_AML_DPD_group_assignment")
GDRIVE_DATASET_7Z = GDRIVE_PROJECT / "DPD_Small.7z"
GDRIVE_RUNS = GDRIVE_PROJECT / "runs"
SYNC_DIR = str(GDRIVE_RUNS)

# --- Fast Local Storage Paths (/content/) ---
LOCAL_ARCHIVE = Path("/content/DPD_Small.7z")
EXTRACT_DEST = Path("/content/dataset")

print("Configuration parameters loaded successfully:")
print(f"  Configs ({len(CONFIG_PATHS)}):")
for i, cfg_p in enumerate(CONFIG_PATHS, 1):
    print(f"    {i}. {cfg_p}")
print(f"  Project Dir:   {GDRIVE_PROJECT}")
print(f"  Sync Dir:      {SYNC_DIR}")
print(f"  Resume:        {RESUME}")
print(f"  Overrides:     {OVERRIDES}")


Configuration parameters loaded successfully:
  Configs (4):
    1. config/multitask/Nat_multitask_convnext_base_v4_1_cutmix.json
    2. config/multitask/Nat_multitask_convnext_base_v4_2_balsoftmax.json
    3. config/multitask/Nat_multitask_convnext_base_v4_3_droppath.json
    4. config/multitask/Nat_multitask_convnext_base_v4_final.json
  Project Dir:   /content/drive/MyDrive/Nat_AML_DPD_group_assignment
  Sync Dir:      /content/drive/MyDrive/Nat_AML_DPD_group_assignment/runs
  Resume:        False
  Overrides:     ['runtime.num_workers=2', 'runtime.checkpoint_every_n_epochs=5']


In [9]:
# Step 2: Mount Google Drive & Verify Project Folder
from google.colab import drive
from pathlib import Path
import os

# Mount Drive
drive.mount('/content/drive')

# Verify paths configured in Step 1 exist
assert GDRIVE_PROJECT.exists(), f"Google Drive folder not found: {GDRIVE_PROJECT}"
assert GDRIVE_DATASET_7Z.exists(), f"Dataset archive not found at: {GDRIVE_DATASET_7Z}"
GDRIVE_RUNS.mkdir(parents=True, exist_ok=True)

print("Google Drive mounted and verified successfully!")
print(f"Project folder: {GDRIVE_PROJECT}")
print(f"Dataset archive: {GDRIVE_DATASET_7Z}")
print(f"Persistent runs folder: {GDRIVE_RUNS}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted and verified successfully!
Project folder: /content/drive/MyDrive/Nat_AML_DPD_group_assignment
Dataset archive: /content/drive/MyDrive/Nat_AML_DPD_group_assignment/DPD_Small.7z
Persistent runs folder: /content/drive/MyDrive/Nat_AML_DPD_group_assignment/runs


In [10]:
# Step 3: Extract Dataset to Fast Local Colab SSD (/content/)
import shutil
from pathlib import Path
import os

EXTRACT_DEST.mkdir(parents=True, exist_ok=True)

# 1. Copy .7z from Google Drive to local SSD (if not already copied)
if not LOCAL_ARCHIVE.exists():
    print(f"Copying {GDRIVE_DATASET_7Z} to local SSD ({LOCAL_ARCHIVE})...")
    shutil.copy2(GDRIVE_DATASET_7Z, LOCAL_ARCHIVE)
    print("Copy complete!")
else:
    print(f"Local archive {LOCAL_ARCHIVE} already present.")

# 2. Extract using 7z if CSVs are not yet present
def find_dpd_root(search_dir: Path) -> Path | None:
    for candidate in [search_dir, search_dir / "DPD_Small", *search_dir.glob("**/DPD_Small*")]:
        if candidate.is_dir() and (candidate / "DPD_Small_train.csv").exists():
            return candidate
    return None

dpd_root = find_dpd_root(EXTRACT_DEST)
if dpd_root is None:
    print("Extracting dataset with 7z (fast multithreaded)...", flush=True)
    !7z x -y /content/DPD_Small.7z -o/content/dataset
    dpd_root = find_dpd_root(EXTRACT_DEST)

assert dpd_root is not None, f"Could not find DPD_Small_train.csv inside {EXTRACT_DEST}!"

# 3. Verify all dataset components exist
assert (dpd_root / "DPD_Small_train.csv").exists(), f"Missing train CSV in {dpd_root}"
assert (dpd_root / "DPD_Small_test_seen.csv").exists(), f"Missing test seen CSV in {dpd_root}"
assert (dpd_root / "DPD_Small_test_unseen.csv").exists(), f"Missing test unseen CSV in {dpd_root}"
assert (dpd_root / "DPD_Small_dataset").exists(), f"Missing DPD_Small_dataset folder in {dpd_root}"

# 4. Export DPD_ROOT environment variable (read by dpd/data.py)
os.environ["DPD_ROOT"] = str(dpd_root)
print(f"\nDataset verified successfully!")
print(f"DPD_ROOT set to: {os.environ['DPD_ROOT']}")

Local archive /content/DPD_Small.7z already present.

Dataset verified successfully!
DPD_ROOT set to: /content/dataset


In [11]:
# Step 4: Install Required Python Packages
!apt-get install -y -qq
!pip install -q pandas pillow tqdm matplotlib


In [12]:
# Step 5: Run Multi-Task CNN Training Sequentially Across All Configs
import subprocess
import sys

total = len(CONFIG_PATHS)
for idx, config_path in enumerate(CONFIG_PATHS, 1):
    print("\n" + "=" * 70)
    print(f"[{idx}/{total}] Starting training for: {config_path}")
    print("=" * 70)

    cmd = [
        sys.executable,
        "-m", "train.train_multi_task_cnn",
        "--config", config_path,
        "--sync-dir", SYNC_DIR,
    ]

    if OVERRIDES:
        cmd.append("--set")
        cmd.extend(OVERRIDES)

    if RESUME:
        cmd.append("--resume")

    print(" " .join(cmd))
    print("-" * 70)

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in proc.stdout:
        print(line, end="", flush=True)

    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Training failed for {config_path} with exit code {proc.returncode}")
    print(f"\n[{idx}/{total}] Finished {config_path} and synced outputs to Google Drive!")

print("\nAll configurations finished training successfully!")



[1/4] Starting training for: config/multitask/Nat_multitask_convnext_base_v4_1_cutmix.json
/usr/bin/python3 -m train.train_multi_task_cnn --config config/multitask/Nat_multitask_convnext_base_v4_1_cutmix.json --sync-dir /content/drive/MyDrive/Nat_AML_DPD_group_assignment/runs --set runtime.num_workers=2 runtime.checkpoint_every_n_epochs=5
----------------------------------------------------------------------
Applied overrides:
  runtime.num_workers=2
  runtime.checkpoint_every_n_epochs=5
Live synchronization enabled -> /content/drive/MyDrive/Nat_AML_DPD_group_assignment/runs/multitask/Nat_multitask_convnext_base_v4_1_cutmix
Gamma tuning disabled (gamma_grid=0): evaluating uncalibrated (gamma=0.0) across all stages.

--- Validation Split 1/3 (fit 5740, val-seen 852, pseudo-unseen 559) ---

[Stage 1 | Split 1/3] >>> Currently Training Epoch 1/30 [backbone frozen]...
  [Split 1/3 Ep 1/30] Step 71/358 | running_loss=9.7541 | crop_acc=3.3% disease_acc=0.7%
  [Split 1/3 Ep 1/30] Step 142/35

In [13]:
# Step 6: Summarize All Configurations and Results
import json
import pandas as pd
from pathlib import Path
from IPython.display import display

rows = []
for config_path in CONFIG_PATHS:
    cfg_file = Path(config_path)
    try:
        run_name = json.loads(cfg_file.read_text(encoding="utf-8")).get("version", cfg_file.stem)
    except Exception:
        run_name = cfg_file.stem

    local_out_dir = Path("runs/multitask") / run_name
    gdrive_out_dir = GDRIVE_RUNS / "multitask" / run_name
    out_dir = gdrive_out_dir if gdrive_out_dir.exists() else local_out_dir

    metrics_file = out_dir / "metrics.json"
    gamma_metrics_file = out_dir / "metrics_gamma.json"

    candidates = []
    if metrics_file.exists():
        candidates.append(("uncalibrated (gamma=0)", json.loads(metrics_file.read_text(encoding="utf-8"))))
    if gamma_metrics_file.exists():
        m_cal = json.loads(gamma_metrics_file.read_text(encoding="utf-8"))
        if float(m_cal.get("gamma", 0.0)) > 0:
            candidates.append((f"calibrated (gamma={m_cal.get('gamma')})", m_cal))

    if not candidates:
        rows.append({
            "version": run_name,
            "eval_mode": "missing",
            "epoch": None,
            "gamma": None,
            "hm_top1": None,
            "hm_top5": None,
            "seen_top1": None,
            "seen_crop": None,
            "seen_disease": None,
            "unseen_top1": None,
            "unseen_crop": None,
            "unseen_disease": None,
            "diag_unseen_only": None,
            "config": config_path,
        })
        continue

    for mode_label, m in candidates:
        ts = m.get("test_seen", m.get("seen", {}))
        tu = m.get("test_unseen", m.get("unseen", {}))
        rows.append({
            "version": m.get("version", run_name),
            "eval_mode": mode_label,
            "epoch": m.get("selected_epoch", m.get("best_epoch")),
            "gamma": m.get("gamma", 0.0),
            "hm_top1": m.get("harmonic_mean_top1"),
            "hm_top5": m.get("harmonic_mean_top5"),
            "seen_top1": ts.get("top1"),
            "seen_crop": ts.get("crop_top1"),
            "seen_disease": ts.get("disease_top1"),
            "unseen_top1": tu.get("top1"),
            "unseen_crop": tu.get("crop_top1"),
            "unseen_disease": tu.get("disease_top1"),
            "diag_unseen_only": m.get("diagnostic_unseen_only_top1"),
            "config": config_path,
        })

df_summary = pd.DataFrame(rows)
pct_cols = ["hm_top1", "hm_top5", "seen_top1", "seen_crop", "seen_disease", "unseen_top1", "unseen_crop", "unseen_disease", "diag_unseen_only"]
df_display = df_summary.copy()
for col in pct_cols:
    if col in df_display.columns:
        df_display[col] = df_display[col].map(lambda x: f"{x:.2%}" if pd.notna(x) else "-")

print("================ Summary of All Configurations ================")
display(df_display)

summary_csv = GDRIVE_RUNS / "multitask" / "summary_multi_config.csv"
if summary_csv.parent.exists():
    df_summary.to_csv(summary_csv, index=False)
    print(f"Saved summary CSV to: {summary_csv}")


================ Summary of All Configurations ================


,version,eval_mode,epoch,gamma,hm_top1,hm_top5,seen_top1,seen_crop,seen_disease,unseen_top1,unseen_crop,unseen_disease,diag_unseen_only,config
0,Nat_multitask_convnext_base_v4_1_cutmix,uncalibrated (gamma=0),7,0.0,12.49%,49.45%,71.80%,94.05%,73.20%,6.84%,76.38%,13.05%,87.15%,config/multitask/Nat_multitask_convnext_base_v...
1,Nat_multitask_convnext_base_v4_2_balsoftmax,uncalibrated (gamma=0),8,0.0,6.34%,38.93%,72.55%,94.25%,73.75%,3.31%,77.35%,7.60%,85.01%,config/multitask/Nat_multitask_convnext_base_v...
2,Nat_multitask_convnext_base_v4_3_droppath,uncalibrated (gamma=0),7,0.0,13.53%,48.52%,72.55%,93.95%,74.00%,7.46%,77.49%,12.78%,84.67%,config/multitask/Nat_multitask_convnext_base_v...
3,Nat_multitask_convnext_base_v4_final,uncalibrated (gamma=0),7,0.0,6.71%,42.02%,70.40%,93.70%,71.90%,3.52%,76.31%,10.08%,86.53%,config/multitask/Nat_multitask_convnext_base_v...


Saved summary CSV to: /content/drive/MyDrive/Nat_AML_DPD_group_assignment/runs/multitask/summary_multi_config.csv
